<a href="https://colab.research.google.com/github/SilkSherstka/hse_machine_learning/blob/main/baseline_traffic_anomaly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Детекция аномалий в веб-трафике: baseline + Isolation Forest

**Проект:** Вариант 4 — детекция аномалий в логах сервера  
**Датасет:** [Web Traffic Time Series](https://www.kaggle.com/datasets/raminhuseyn/web-traffic-time-series-dataset)  
**Подход:** статистический baseline (Z-score и IQR на скользящем окне)

**Гипотеза:** резкие всплески и падения `TrafficCount` можно обнаружить простым правилом — если значение сильно отклоняется от недавней «нормы» (скользящее среднее ± N·σ или границы IQR), это аномалия.

## 1. Установка и импорты

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, precision_recall_fscore_support

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 10)

RANDOM_STATE = 42

## 2. Загрузка данных

Выберите один из способов:
- **Локально / в репозитории:** положите `web_traffic.csv` рядом с ноутбуком.
- **Colab:** загрузите файл через виджет ниже или скачайте с Kaggle.

In [ ]:
from pathlib import Path

DATA_PATH = Path("web_traffic.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("Файл не найден. Загрузите web_traffic.csv:")
        uploaded = files.upload()
        DATA_PATH = Path(list(uploaded.keys())[0])
    except ImportError:
        raise FileNotFoundError(
            "Положите web_traffic.csv в ту же папку, что и ноутбук, "
            "или откройте ноутбук в Google Colab и загрузите файл."
        )

df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

print(f"Строк: {len(df)}")
df.head()

## 3. Мини-EDA

In [ ]:
print(df.dtypes)
print()
df.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["Timestamp"], df["TrafficCount"], linewidth=0.8)
ax.set_title("Веб-трафик во времени")
ax.set_xlabel("Timestamp")
ax.set_ylabel("TrafficCount")
plt.tight_layout()
plt.show()

## 4. Параметры baseline

| Параметр | Значение | Смысл |
|----------|----------|-------|
| `WINDOW` | 24 | Скользящее окно (~сутки при интервале ~1 ч) |
| `N_SIGMA` | 3 | Порог Z-score: аномалия, если \|z\| > 3 |
| IQR множитель | 1.5 | Классическое правило «усов» boxplot |

In [ ]:
WINDOW = 24
N_SIGMA = 3
IQR_MULTIPLIER = 1.5

## 5. Baseline 1: Z-score

Для каждой точки:

$$z_t = \frac{x_t - \mu_t}{\sigma_t}$$

где $\mu_t$ и $\sigma_t$ — скользящее среднее и стандартное отклонение `TrafficCount` за последние `WINDOW` интервалов.

**Правило:** `is_anomaly = 1`, если $|z_t| > N\_SIGMA$.

In [ ]:
df["rolling_mean"] = df["TrafficCount"].rolling(WINDOW, min_periods=WINDOW).mean()
df["rolling_std"] = df["TrafficCount"].rolling(WINDOW, min_periods=WINDOW).std()
df["z_score"] = (df["TrafficCount"] - df["rolling_mean"]) / df["rolling_std"]
df["is_anomaly_zscore"] = df["z_score"].abs() > N_SIGMA

valid = df.dropna(subset=["z_score"])
n_z = valid["is_anomaly_zscore"].sum()
print(f"Z-score: {n_z} аномалий из {len(valid)} ({100 * n_z / len(valid):.1f}%)")

valid.loc[valid["is_anomaly_zscore"], ["Timestamp", "TrafficCount", "rolling_mean", "z_score"]].head(10)

## 6. Baseline 2: IQR

В том же окне считаем квартили Q1, Q3 и IQR = Q3 − Q1.

**Границы нормы:**  
`low = Q1 - 1.5 * IQR`, `high = Q3 + 1.5 * IQR`

**Правило:** аномалия, если значение ниже `low` или выше `high`.

In [ ]:
def iqr_anomaly_flags(series: pd.Series, window: int, multiplier: float = 1.5) -> pd.Series:
    """Флаг аномалии по IQR в скользящем окне."""
    flags = np.zeros(len(series), dtype=bool)
    values = series.to_numpy()

    for i in range(window - 1, len(values)):
        w = values[i - window + 1 : i + 1]
        q1, q3 = np.percentile(w, [25, 75])
        iqr = q3 - q1
        low = q1 - multiplier * iqr
        high = q3 + multiplier * iqr
        flags[i] = (values[i] < low) or (values[i] > high)

    return pd.Series(flags, index=series.index)


df["is_anomaly_iqr"] = iqr_anomaly_flags(df["TrafficCount"], WINDOW, IQR_MULTIPLIER)

n_iqr = df["is_anomaly_iqr"].sum()
print(f"IQR: {n_iqr} аномалий из {len(df)} ({100 * n_iqr / len(df):.1f}%)")

df.loc[df["is_anomaly_iqr"], ["Timestamp", "TrafficCount"]].head(10)

## 7. Сравнение методов

In [ ]:
z_flags = df["is_anomaly_zscore"].fillna(False)
iqr_flags = df["is_anomaly_iqr"]

both = (z_flags & iqr_flags).sum()
only_z = (z_flags & ~iqr_flags).sum()
only_iqr = (~z_flags & iqr_flags).sum()

summary = pd.DataFrame(
    {
        "Метод": ["Z-score", "IQR", "Оба согласны", "Только Z-score", "Только IQR"],
        "Кол-во": [n_z, n_iqr, both, only_z, only_iqr],
    }
)
summary

## 8. Визуализация аномалий

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, col, title, color in [
    (axes[0], "is_anomaly_zscore", f"Z-score (N={N_SIGMA})", "red"),
    (axes[1], "is_anomaly_iqr", "IQR", "darkorange"),
]:
    ax.plot(df["Timestamp"], df["TrafficCount"], linewidth=0.6, alpha=0.7, label="TrafficCount")
    mask = df[col].fillna(False)
    ax.scatter(
        df.loc[mask, "Timestamp"],
        df.loc[mask, "TrafficCount"],
        color=color,
        s=25,
        label="Аномалия",
        zorder=3,
    )
    ax.set_title(title)
    ax.set_ylabel("TrafficCount")
    ax.legend(loc="upper right")

axes[1].set_xlabel("Timestamp")
plt.tight_layout()
plt.show()

## 9. Выводы

- **Z-score** — строже: ловит только сильные отклонения от локального среднего.
- **IQR** — мягче: больше точек помечается как аномальные, устойчивее к отдельным выбросам в окне.
- Оба метода не требуют обучения модели и служат **baseline** для сравнения с Isolation Forest.
- Следующий шаг: добавить признаки (lag, hour, rolling stats) и сравнить F1 с улучшенной моделью.